In [ ]:
import numpy as np, pandas as pd, os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Experiment B1 — ZK Failure-Mode Taxonomy

The report says a system with high CIC2 but low RC is 'one-way signalling rather than genuine communication.' B1 trains 5 games with DIFFERENT broken causal links (full communication, receiver ignores M1, sender ignores query, receiver ignores M2, severed channel) and verifies that each produces a distinct (CIC1, RC, CIC2) signature — turning the ZK metric into a diagnostic taxonomy.

In [ ]:
!pip install torch matplotlib pandas -q

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
torch.manual_seed(42)

# ── The report's claim being tested ─────────────────────────────────────────
# "a system with high CIC2 but low RC is one-way signalling rather than genuine
# communication." (ZK section of cross-experiment discussion)
# B1 builds a taxonomy by training 5 games with DIFFERENT broken links:
#   Mode 0: Full communication (baseline — all links intact)
#   Mode 1: Receiver ignores M1 (high CIC2, zero CIC1, zero RC)
#   Mode 2: Sender ignores query (high CIC1, zero RC, zero CIC2)
#   Mode 3: Receiver ignores M2 (high CIC1, zero CIC2, low RC)
#   Mode 4: Severed channel (all zero — original Exp 2 baseline)
# Each mode should produce a distinct (CIC1, RC, CIC2) signature.

N_FEATURES   = 16
N_CANDIDATES = 5
VOCAB_SIZE   = 16
QUERY_DIM    = 16
HIDDEN       = 64
N_TRAIN      = 8000
N_VAL        = 2000
N_EPOCHS     = 80
BATCH_SIZE   = 128
N_SCRAMBLES  = 20
print('Setup complete. Building 5 failure-mode games...')

In [ ]:
def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    objs = rng.integers(0,2,(n,N_CANDIDATES,N_FEATURES)).astype(np.float32)
    return TensorDataset(torch.tensor(objs[:,0,:]), torch.tensor(objs), torch.zeros(n,dtype=torch.long))

train_ds = make_dataset(N_TRAIN, 1)
val_ds   = make_dataset(N_VAL, 2)

class Sender(nn.Module):
    def __init__(self):
        super().__init__()
        self.turn1 = nn.Sequential(nn.Linear(N_FEATURES,HIDDEN),nn.ReLU(),nn.Linear(HIDDEN,VOCAB_SIZE))
        self.turn2 = nn.Sequential(nn.Linear(N_FEATURES+QUERY_DIM,HIDDEN),nn.ReLU(),nn.Linear(HIDDEN,VOCAB_SIZE))
    def forward_t1(self, obj): return self.turn1(obj)
    def forward_t2(self, obj, query): return self.turn2(torch.cat([obj,query],-1))

class Receiver(nn.Module):
    def __init__(self):
        super().__init__()
        self.msg_proj  = nn.Linear(VOCAB_SIZE, HIDDEN)
        self.obj_proj  = nn.Linear(N_FEATURES, HIDDEN)
        self.query_mlp = nn.Sequential(nn.Linear(HIDDEN,HIDDEN),nn.ReLU(),nn.Linear(HIDDEN,QUERY_DIM),nn.Tanh())
    def attend(self, msg_h, cands):
        return torch.bmm(self.obj_proj(cands), msg_h.unsqueeze(-1)).squeeze(-1)
    def forward_t1(self, msg, cands):
        h = self.msg_proj(msg)
        return self.query_mlp(h), self.attend(h, cands)
    def forward_t2(self, msg1, msg2, cands):
        return self.attend((self.msg_proj(msg1)+self.msg_proj(msg2))/2, cands)

def train_game(mode, n_epochs=N_EPOCHS):
    """
    mode 0: full communication
    mode 1: receiver ignores M1 (query is zeroed)
    mode 2: sender ignores query (M2 = copy of M1)
    mode 3: receiver ignores M2 (final prediction from M1 only)
    mode 4: severed channel (M1=M2=zero)
    """
    s = Sender(); r = Receiver()
    opt = torch.optim.Adam(list(s.parameters())+list(r.parameters()), lr=1e-3)
    tau = 0.5
    for ep in range(n_epochs):
        s.train(); r.train()
        for si, cands, labels in DataLoader(train_ds, BATCH_SIZE, shuffle=True):
            B = si.shape[0]
            logits1 = s.forward_t1(si)
            if mode == 4:  # severed
                msg1 = torch.zeros(B, VOCAB_SIZE)
            else:
                msg1 = F.gumbel_softmax(logits1, tau=tau, hard=True)
            if mode == 1:  # receiver ignores M1: zero the query
                query = torch.zeros(B, QUERY_DIM)
                scores1 = r.attend(r.msg_proj(msg1), cands)
            else:
                query, scores1 = r.forward_t1(msg1, cands)
            if mode == 2:  # sender ignores query: M2 = repeat of M1
                logits2 = logits1.detach()
            elif mode == 4:
                logits2 = logits1  # doesn't matter, msg2 is zeroed
            else:
                logits2 = s.forward_t2(si, query)
            if mode == 4:
                msg2 = torch.zeros(B, VOCAB_SIZE)
            else:
                msg2 = F.gumbel_softmax(logits2, tau=tau, hard=True)
            if mode == 3:  # receiver ignores M2: predict from M1 only
                scores2 = scores1
            else:
                scores2 = r.forward_t2(msg1, msg2, cands)
            loss = F.cross_entropy(scores2, labels)
            loss.backward(); opt.step(); opt.zero_grad()
    return s, r

games = {}
MODE_NAMES = {
    0: 'Full communication',
    1: 'Receiver ignores M1',
    2: 'Sender ignores query',
    3: 'Receiver ignores M2',
    4: 'Severed channel',
}
for mode, name in MODE_NAMES.items():
    print(f'Training mode {mode}: {name}...')
    games[mode] = train_game(mode)
print('All 5 games trained')

In [ ]:
def js_divergence(p_logits, q_logits):
    p = F.softmax(p_logits, dim=-1); q = F.softmax(q_logits, dim=-1)
    m = 0.5*(p+q)
    return (0.5*(p*(p/m.clamp(1e-9)).log()).sum(-1)+0.5*(q*(q/m.clamp(1e-9)).log()).sum(-1)).mean().item()

def compute_zk(sender, receiver):
    sender.eval(); receiver.eval()
    cic1_all, rc_all, cic2_all = [], [], []
    rng = np.random.default_rng(7)
    with torch.no_grad():
        for si, cands, labels in DataLoader(val_ds, 256):
            B = si.shape[0]
            logits1 = sender.forward_t1(si)
            msg1_real = F.one_hot(logits1.argmax(-1), VOCAB_SIZE).float()
            query_real, _ = receiver.forward_t1(msg1_real, cands)
            logits2_real = sender.forward_t2(si, query_real)
            msg2_real = F.one_hot(logits2_real.argmax(-1), VOCAB_SIZE).float()
            scores2_real = receiver.forward_t2(msg1_real, msg2_real, cands)
            for _ in range(N_SCRAMBLES):
                perm = torch.randperm(B)
                msg1_fake = msg1_real[perm]
                query_fake, _ = receiver.forward_t1(msg1_fake, cands)
                cic1_all.append((query_real-query_fake).pow(2).sum(-1).sqrt().mean().item())
                logits2_fake = sender.forward_t2(si, query_fake)
                rc_all.append(js_divergence(logits2_real, logits2_fake))
                msg2_fake = msg2_real[perm]
                scores2_fake = receiver.forward_t2(msg1_real, msg2_fake, cands)
                cic2_all.append((scores2_real-scores2_fake).pow(2).sum(-1).sqrt().mean().item())
    cic1=np.mean(cic1_all); rc=np.mean(rc_all); cic2=np.mean(cic2_all)
    return {'CIC1':round(cic1,4),'RC':round(rc,5),'CIC2':round(cic2,4),'ZK':round((cic1+cic2)/2*rc,5)}

def eval_acc(sender, receiver, mode):
    sender.eval(); receiver.eval(); accs=[]
    with torch.no_grad():
        for si, cands, labels in DataLoader(val_ds, 256):
            B=si.shape[0]
            msg1=F.one_hot(sender.forward_t1(si).argmax(-1),VOCAB_SIZE).float()
            if mode==1: query=torch.zeros(B,QUERY_DIM)
            else: query,_=receiver.forward_t1(msg1,cands)
            if mode==2: logits2=sender.forward_t1(si)
            else: logits2=sender.forward_t2(si,query)
            msg2=F.one_hot(logits2.argmax(-1),VOCAB_SIZE).float()
            if mode==3: scores2=receiver.attend(receiver.msg_proj(msg1),cands)
            elif mode==4: scores2=receiver.forward_t2(torch.zeros_like(msg1),torch.zeros_like(msg2),cands)
            else: scores2=receiver.forward_t2(msg1,msg2,cands)
            accs.append((scores2.argmax(-1)==labels).float().mean().item())
    return float(np.mean(accs))

print('Computing ZK metrics for all modes...')
all_metrics = {}
for mode, name in MODE_NAMES.items():
    s, r = games[mode]
    zk = compute_zk(s, r)
    acc = eval_acc(s, r, mode)
    all_metrics[mode] = {**zk, 'accuracy':round(acc,4), 'name':name}
    print(f'  Mode {mode} ({name}): acc={acc:.3f} ZK={zk["ZK"]:.5f} CIC1={zk["CIC1"]:.4f} RC={zk["RC"]:.5f} CIC2={zk["CIC2"]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

modes = list(all_metrics.keys())
names = [all_metrics[m]['name'] for m in modes]
cic1  = [all_metrics[m]['CIC1'] for m in modes]
rc    = [all_metrics[m]['RC']   for m in modes]
cic2  = [all_metrics[m]['CIC2'] for m in modes]
zk    = [all_metrics[m]['ZK']   for m in modes]
acc   = [all_metrics[m]['accuracy'] for m in modes]

# Panel 1: Component stacked bars (the diagnostic lookup table visualised)
ax = axes[0]
x = np.arange(len(modes))
# Normalise each component to [0,1] for comparison
max_c1=max(cic1)+1e-9; max_rc=max(rc)+1e-9; max_c2=max(cic2)+1e-9
ax.bar(x-0.25, [v/max_c1 for v in cic1], 0.2, label='CIC1 (norm)', color='#378ADD')
ax.bar(x-0.05, [v/max_rc  for v in rc],   0.2, label='RC (norm)',   color='#1D9E75')
ax.bar(x+0.15, [v/max_c2  for v in cic2], 0.2, label='CIC2 (norm)', color='#EF9F27')
ax.bar(x+0.35, [v/max(zk+[1e-9]) for v in zk], 0.2, label='ZK (norm)',  color='#E24B4A')
ax.set_xticks(x); ax.set_xticklabels([f'Mode {m}' for m in modes], rotation=30, ha='right')
ax.set_ylabel('Normalised component value')
ax.set_title('ZK component signatures per failure mode\n(distinct patterns = taxonomy works)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# Panel 2: Heatmap of component presence/absence — the diagnostic lookup table
ax = axes[1]
import matplotlib.colors as mcolors
comp_names = ['CIC1', 'RC', 'CIC2', 'ZK', 'Task acc']
comp_vals = np.array([
    [all_metrics[m]['CIC1'], all_metrics[m]['RC'], all_metrics[m]['CIC2'],
     all_metrics[m]['ZK'], all_metrics[m]['accuracy']]
    for m in modes
])
# Normalise each column
col_max = comp_vals.max(axis=0).clip(1e-9)
comp_norm = comp_vals / col_max
im = ax.imshow(comp_norm, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(comp_names))); ax.set_xticklabels(comp_names)
ax.set_yticks(range(len(modes)))
ax.set_yticklabels([f'Mode {m}: {all_metrics[m]["name"]}' for m in modes], fontsize=9)
for i in range(len(modes)):
    for j in range(len(comp_names)):
        ax.text(j, i, f'{comp_vals[i,j]:.3f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, label='Normalised value')
ax.set_title('Diagnostic lookup table\n(ZK component signatures by failure mode)')

plt.suptitle('Experiment B1 — ZK Failure-Mode Taxonomy\n'
             'Each failure mode produces a distinct (CIC1, RC, CIC2) signature',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp_b1_zk_taxonomy.png', dpi=150, bbox_inches='tight')
plt.show()

import pandas as pd
print('\n' + '='*70)
print('EXPERIMENT B1 — ZK FAILURE-MODE TAXONOMY')
print('='*70)
print(pd.DataFrame(all_metrics).T[['name','accuracy','CIC1','RC','CIC2','ZK']].to_string())
print()
# Check whether each mode is distinguishable
print('\nDistinguishability check (do modes produce distinct ZK signatures?):')
for mode, metrics in all_metrics.items():
    sig = []
    if metrics['CIC1'] > 0.5 * all_metrics[0]['CIC1']: sig.append('CIC1+')
    else: sig.append('CIC1~0')
    if metrics['RC'] > 0.5 * all_metrics[0]['RC']: sig.append('RC+')
    else: sig.append('RC~0')
    if metrics['CIC2'] > 0.5 * all_metrics[0]['CIC2']: sig.append('CIC2+')
    else: sig.append('CIC2~0')
    print(f'  Mode {mode} ({metrics["name"]}): {" | ".join(sig)}  ZK={metrics["ZK"]:.5f}')